# UPR Bootstrap — Sync Latest `upr/` Code from GitHub

**Run once per Colab session before Notebooks 01–04.**

In [ ]:
import os
import sys
import shutil
import subprocess
import importlib

GITHUB_REPO = 'https://github.com/chiragrohit/UniversalPrecisionRuntime.git'
TMP_CLONE   = '/content/upr_tmp'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
os.makedirs(PROJECT_DIR, exist_ok=True)

# 1. Clone repo to ephemeral /content/ (not Drive)
print('Cloning latest code from GitHub...')
if os.path.exists(TMP_CLONE):
    shutil.rmtree(TMP_CLONE)

result = subprocess.run(
    ['git', 'clone', '--depth=1', GITHUB_REPO, TMP_CLONE],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('git clone failed:', result.stderr)
else:
    print('Clone successful.')

# 2. Delete old __pycache__ from Drive upr/ BEFORE copying
#    (stale .pyc files cause Python to ignore the new .py files)
dest_upr   = os.path.join(PROJECT_DIR, 'upr')
pycache_dir = os.path.join(dest_upr, '__pycache__')
if os.path.exists(pycache_dir):
    shutil.rmtree(pycache_dir)
    print('Deleted stale __pycache__ from Drive.')

# 3. Copy upr/ from clone → Drive (overwrites all .py files)
src_upr = os.path.join(TMP_CLONE, 'upr')
if os.path.exists(src_upr):
    # Remove any __pycache__ inside the clone too
    clone_pycache = os.path.join(src_upr, '__pycache__')
    if os.path.exists(clone_pycache):
        shutil.rmtree(clone_pycache)

    shutil.copytree(src_upr, dest_upr, dirs_exist_ok=True)
    print(f'Copied upr/ → {dest_upr}')
    for f in sorted(os.listdir(dest_upr)):
        print(f'  ✓ upr/{f}')
else:
    print('ERROR: upr/ not found in cloned repo.')

# 4. Cleanup temp clone
shutil.rmtree(TMP_CLONE, ignore_errors=True)

# 5. Force Python to forget the old cached upr module
for mod_name in list(sys.modules.keys()):
    if mod_name == 'upr' or mod_name.startswith('upr.'):
        del sys.modules[mod_name]
importlib.invalidate_caches()

# 6. Add project dir to path and import fresh
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import upr
upr.set_seed(42)
print(f'\nupr {upr.__version__} imported successfully — ready to run Notebooks 01 → 04.')